In [0]:
raw_table = dbutils.widgets.get("raw_table")
landing_table = dbutils.widgets.get("landing_table")

In [0]:
display(
spark.sql(f"""
INSERT INTO {raw_table}
SELECT
  'BEARS' AS source_system,
  payer AS payer,
  CAST(ofc AS INT) AS office_number,
  CASE
      WHEN
          SUM(CAST(regexp_replace(applied_on_mr, '[,$]', '') AS DOUBLE)) = 0
          AND SUM(CAST(regexp_replace(applied, '[,$]', '') AS DOUBLE)) <> 0
      THEN SUM(CAST(regexp_replace(applied, '[,$]', '') AS DOUBLE))
      ELSE SUM(CAST(regexp_replace(applied_on_mr, '[,$]', '') AS DOUBLE))
  END AS cash_collected,
  bill_to_num AS payor_id,
  bill_to_name AS bill_to_name,
  client_num AS client_number,
  try_to_date(posted_dt, 'MM-dd-yy') AS posted_date,
  try_to_date(deposit_dt, 'MM-dd-yy') AS deposit_date,
  batch_id AS batch_id,
  check_id AS check_id,
  invoice_num AS invoice_number,
  type AS type,
  batch_num AS batch_number,
  bank AS bank,
  product AS product,
  try_to_date(RIGHT(_file_name, 6), 'MMddyy') AS date_entered,
  payment_number AS payment_number,
  _load_timestamp AS _load_timestamp,
  _file_name AS _file_name
FROM {landing_table} landing
WHERE NOT EXISTS (
    SELECT 1
    FROM {raw_table} raw
    WHERE raw._load_timestamp = landing._load_timestamp
    AND raw._file_name = landing._file_name 
)
GROUP BY
    payer, ofc, bill_to_num, bill_to_name, client_num,
    posted_dt, deposit_dt, batch_id, check_id,
    invoice_num, type, batch_num, bank,
    product, date_entered, payment_number, _load_timestamp, _file_name
""")
)